# Badminton Action Classification — GPU Pipeline (Google Colab)

Runs the full pipeline on a CUDA GPU: **video → RTMPose keypoints → 5-fold CV → final model**.
On a free T4, extracting all 662 clips takes a few minutes (vs ~2 hrs on CPU).

### Before you start
1. **Runtime → Change runtime type → Hardware accelerator = GPU.**
2. Put the project + dataset on Google Drive, e.g.:
   - Code:  `MyDrive/Action_classification/`  (the repo: `src/`, `scripts/`, `configs/`, `pyproject.toml`)
   - Videos: `MyDrive/Action_classification/data/archive/<class>/*.mp4`
   (Zipping the repo and uploading is easiest; unzip in Drive.)

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive and point at the project
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Action_classification'  # <-- edit if different
DATA_ARCHIVE = os.path.join(PROJECT_DIR, 'data/archive')
assert os.path.isdir(PROJECT_DIR), f'project not found: {PROJECT_DIR}'
assert os.path.isdir(DATA_ARCHIVE), f'videos not found: {DATA_ARCHIVE}'
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())
print('classes:', sorted(os.listdir(DATA_ARCHIVE)))

In [ ]:
# 3. Install deps. Three gotchas, all handled here:
#    (a) rtmlib pulls in CPU onnxruntime, which shadows the GPU build -> install rtmlib
#        first, then REPLACE onnxruntime with a GPU build.
#    (b) PyPI onnxruntime-gpu may be built for CUDA 13, but Colab runs CUDA 12 -> use the
#        CUDA-12 build from Microsoft's official index.
#    (c) bare `!python`/`!pip` can point at a DIFFERENT interpreter than the kernel, so the
#        package "isn't found" later. Always drive pip/python via sys.executable.
import sys
!{sys.executable} -m pip install -q rtmlib opencv-python-headless
!{sys.executable} -m pip install -q -e .
!{sys.executable} -m pip uninstall -y -q onnxruntime onnxruntime-gpu
!{sys.executable} -m pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
# Fallback if the index is unreachable:  !{sys.executable} -m pip install -q onnxruntime-gpu==1.19.2
!ls /usr/local/cuda/lib64/libcudart.so.* 2>/dev/null || echo 'no cuda libs -> is the runtime set to GPU?'
print('\n>>> Now RESTART the runtime (Runtime -> Restart session), then run cell 3b and onward.')

In [ ]:
# 3b. RUN AFTER RESTART. Verify onnxruntime sees the GPU before extracting.
#     Also re-set the working directory (restart resets it).
import os
PROJECT_DIR = '/content/drive/MyDrive/Action_classification'  # same as cell 2
os.chdir(PROJECT_DIR)

import onnxruntime as ort
providers = ort.get_available_providers()
print('ORT providers:', providers)
assert 'CUDAExecutionProvider' in providers, (
    "CUDA provider missing. Fallback: !pip install onnxruntime-gpu==1.19.2 "
    "(match Colab's CUDA), then restart again."
)
print('GPU ready ✔')

In [ ]:
# 4. Extract keypoints on GPU  (whole-clip sampling, 24 frames/clip)
#    Writes data/kp/*.npy + data/manifest.jsonl under the project dir.
import sys
!{sys.executable} scripts/extract_keypoints.py \
    --root data/archive --out data \
    --mode balanced --frames 24 --device cuda

In [ ]:
# 5. Cross-validate (GPU training; config device=auto picks CUDA)
import sys
!{sys.executable} -m badminton.training.cross_validate --config configs/default.yaml --folds 5

In [ ]:
# 6. Train the final model -> model.pt
import sys
!{sys.executable} -m badminton.training.train --config configs/default.yaml

In [ ]:
# 7. Save the checkpoint back to Drive (and offer a direct download)
import shutil
shutil.copy('model.pt', os.path.join(PROJECT_DIR, 'model.pt'))
print('saved model.pt to Drive')
from google.colab import files
files.download('model.pt')

### Notes
- `--device cuda` routes RTMPose through `CUDAExecutionProvider`; training auto-detects CUDA.
- For an even faster smoke test, add `--per-class 20` to the extraction cell.
- Re-running extraction overwrites `data/kp/` and `data/manifest.jsonl`.
- The same code runs locally on CPU (`make extract && make cv`) or Apple-silicon MPS for training (`device: mps`).